In [ ]:

# =============================================================================
# Angle Distribution & Time Series Plotter
# =============================================================================
# Plots two angles.csv files as:
#   1. KDE distribution curves (overlaid)
#   2. Timestep vs angle time series (overlaid)
#
# Configure the parameters in the cell below, then Run All.
# =============================================================================

# ── USER PARAMETERS ──────────────────────────────────────────────────────────

FILE_A      = "close_angle.csv"   # path to first  angles.csv
FILE_B      = "open_angle.csv"   # path to second angles.csv

LABEL_A     = "close"   # legend label for file A
LABEL_B     = "open"   # legend label for file B

START_STEP  = 0                # only use frames with timestep >= this value
                               # set to 0 to include all frames

KDE_BW      = 0.5              # KDE bandwidth (smoothness of distribution curve)
                               # larger = smoother; try 0.1 – 2.0

SAVE_FIG    = True             # set False to only display, not save
FIG_FORMAT  = "svg"            # "pdf" | "png" | "svg" | "eps"
DPI         = 300              # resolution (relevant for png)

# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# ── Load & filter data ───────────────────────────────────────────────────────

def load(path, start_step):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    df = df[df["timestep"] >= start_step].reset_index(drop=True)
    print(f"  {path}: {len(df)} frames after filtering (start_step={start_step})")
    return df

print("Loading data...")
df_a = load(FILE_A, START_STEP)
df_b = load(FILE_B, START_STEP)

angles_a = df_a["angle_deg"].values
angles_b = df_b["angle_deg"].values

# ── Shared x-axis range for KDE ──────────────────────────────────────────────

x_min = max(0,   min(angles_a.min(), angles_b.min()) - 5)
x_max = min(180, max(angles_a.max(), angles_b.max()) + 5)
x     = np.linspace(x_min, x_max, 500)

kde_a = gaussian_kde(angles_a, bw_method=KDE_BW)
kde_b = gaussian_kde(angles_b, bw_method=KDE_BW)

# ── Plot 1: Distribution ─────────────────────────────────────────────────────

fig1, ax1 = plt.subplots(figsize=(7, 4))

ax1.plot(x, kde_a(x), label=LABEL_A, linewidth=2)
ax1.fill_between(x, kde_a(x), alpha=0.15)

ax1.plot(x, kde_b(x), label=LABEL_B, linewidth=2)
ax1.fill_between(x, kde_b(x), alpha=0.15)

ax1.set_xlabel("Angle (°)", fontsize=12)
ax1.set_ylabel("Probability density", fontsize=12)
ax1.set_title("Angle distribution", fontsize=13)
ax1.legend(fontsize=11)
ax1.set_xlim(x_min, x_max)
ax1.set_ylim(bottom=0)
fig1.tight_layout()

if SAVE_FIG:
    out1 = f"angle_distribution.{FIG_FORMAT}"
    fig1.savefig(out1, dpi=DPI, bbox_inches="tight")
    print(f"Saved: {out1}")

plt.show()

# ── Plot 2: Time series ───────────────────────────────────────────────────────

fig2, ax2 = plt.subplots(figsize=(9, 4))

ax2.plot(df_a["timestep"], angles_a, label=LABEL_A, linewidth=0.8, alpha=0.85)
ax2.plot(df_b["timestep"], angles_b, label=LABEL_B, linewidth=0.8, alpha=0.85)

ax2.set_xlabel("Timestep", fontsize=12)
ax2.set_ylabel("Angle (°)", fontsize=12)
ax2.set_title("Angle over time", fontsize=13)
ax2.legend(fontsize=11)
ax2.set_ylim(0, 180)
fig2.tight_layout()

if SAVE_FIG:
    out2 = f"angle_timeseries.{FIG_FORMAT}"
    fig2.savefig(out2, dpi=DPI, bbox_inches="tight")
    print(f"Saved: {out2}")

plt.show()
